# Customer Feature Engineering

## Import Libraries

In [2]:
import pandas as pd
import numpy as np
from datetime import timedelta

## Load Data

In [3]:
train_df = pd.read_csv("../data/processed/train_clean.csv")
validation_df = pd.read_csv("../data/processed/validation_clean.csv")

In [4]:
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Train shape: (386732, 9)
Validation shape: (375666, 9)


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 386732 entries, 0 to 386731
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      386732 non-null  int64  
 1   StockCode    386732 non-null  object 
 2   Description  386732 non-null  object 
 3   Quantity     386732 non-null  int64  
 4   InvoiceDate  386732 non-null  object 
 5   Price        386732 non-null  float64
 6   Customer ID  386732 non-null  int64  
 7   Country      386732 non-null  object 
 8   TotalPrice   386732 non-null  float64
dtypes: float64(2), int64(3), object(4)
memory usage: 26.6+ MB


In [6]:
validation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 375666 entries, 0 to 375665
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      375666 non-null  int64  
 1   StockCode    375666 non-null  object 
 2   Description  375666 non-null  object 
 3   Quantity     375666 non-null  int64  
 4   InvoiceDate  375666 non-null  object 
 5   Price        375666 non-null  float64
 6   Customer ID  375666 non-null  int64  
 7   Country      375666 non-null  object 
 8   TotalPrice   375666 non-null  float64
dtypes: float64(2), int64(3), object(4)
memory usage: 25.8+ MB


In [7]:
train_df["InvoiceDate"] = pd.to_datetime(train_df["InvoiceDate"])
validation_df["InvoiceDate"] = pd.to_datetime(validation_df["InvoiceDate"])

In [8]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 386732 entries, 0 to 386731
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      386732 non-null  int64         
 1   StockCode    386732 non-null  object        
 2   Description  386732 non-null  object        
 3   Quantity     386732 non-null  int64         
 4   InvoiceDate  386732 non-null  datetime64[ns]
 5   Price        386732 non-null  float64       
 6   Customer ID  386732 non-null  int64         
 7   Country      386732 non-null  object        
 8   TotalPrice   386732 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(3)
memory usage: 26.6+ MB


In [9]:
validation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 375666 entries, 0 to 375665
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      375666 non-null  int64         
 1   StockCode    375666 non-null  object        
 2   Description  375666 non-null  object        
 3   Quantity     375666 non-null  int64         
 4   InvoiceDate  375666 non-null  datetime64[ns]
 5   Price        375666 non-null  float64       
 6   Customer ID  375666 non-null  int64         
 7   Country      375666 non-null  object        
 8   TotalPrice   375666 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(3)
memory usage: 25.8+ MB


## Define Snapshot Dates

In [10]:
train_snapshot = train_df["InvoiceDate"].max() + timedelta(days=1)
valid_snapshot = validation_df["InvoiceDate"].max() + timedelta(days=1)

print(train_snapshot, valid_snapshot)

2010-12-01 19:35:00 2011-12-01 17:37:00


## Build a Reusable Feature Function

In [12]:
def build_customer_features(df, snapshot_date):
    
    df = df.copy()
    
    # -Basic Aggregations -
    customer = df.groupby("Customer ID").agg({
        "InvoiceDate": ["min", "max"],
        "Invoice": "nunique",
        "TotalPrice": "sum"
    })
    
    customer.columns = [
        "first_purchase",
        "last_purchase",
        "frequency",
        "monetary"  
    ]
    
    customer = customer.reset_index()
    
    # - Recency -
    customer["recency"] = (
        snapshot_date - customer["last_purchase"]
    ).dt.days
    
    # - Customer Lifespan -
    customer["lifespan"] = (
        customer["last_purchase"] - customer["first_purchase"]
    ).dt.days
    
    # - Average Order Value -
    customer["avg_order_value"] = (
        customer["monetary"] / customer["frequency"]
    )
    
    # - Active Months -
    df["year_month"] = df["InvoiceDate"].dt.to_period("M")
    
    active_months = (
        df.groupby("Customer ID")["year_month"]
          .nunique()
          .reset_index(name="active_months")
    )
    
    customer = customer.merge(active_months, on="Customer ID", how="left")
    
    # - Purchase Regularity -
    invoice_dates = (
    df.groupby(["Customer ID", "Invoice"])["InvoiceDate"]
      .min()
      .reset_index()
      .sort_values(["Customer ID", "InvoiceDate"])
    )

    invoice_dates["prev_purchase"] = (
    invoice_dates.groupby("Customer ID")["InvoiceDate"].shift(1)
    )

    invoice_dates["gap_days"] = (
    invoice_dates["InvoiceDate"] - invoice_dates["prev_purchase"]
    ).dt.days

    regularity = (
    invoice_dates.groupby("Customer ID")["gap_days"]
      .mean()
      .reset_index(name="avg_purchase_gap")
    )
    
    customer = customer.merge(regularity, on="Customer ID", how="left")
    
    # - Spending Volatility -
    invoice_level = (
        df.groupby(["Customer ID", "Invoice"])["TotalPrice"]
          .sum()
          .reset_index()
    )
    
    volatility = (
        invoice_level.groupby("Customer ID")["TotalPrice"]
          .std()
          .reset_index(name="spending_volatility")
    )
    
    customer = customer.merge(volatility, on="Customer ID", how="left")
    
    # - Average Basket Size -
    basket_per_invoice = (
        df.groupby(["Customer ID", "Invoice"])["Quantity"]
          .sum()
          .reset_index(name="invoice_item_count")
    )
    
    avg_basket = (
        basket_per_invoice.groupby("Customer ID")["invoice_item_count"]
          .mean()
          .reset_index(name="avg_basket_size")
    )
    
    customer = customer.merge(avg_basket, on="Customer ID", how="left")
    
    # - Revenue Contribution -
    total_revenue = customer["monetary"].sum()
    customer["revenue_contribution"] = (
        customer["monetary"] / total_revenue
    )
    
    # - Fill NA for single purchase customers -
    customer["avg_purchase_gap"] = customer["avg_purchase_gap"].fillna(0)
    customer["spending_volatility"] = customer["spending_volatility"].fillna(0)
    
    return customer


## Generate Train & Validation Features

In [13]:
train_features = build_customer_features(train_df, train_snapshot)
valid_features = build_customer_features(validation_df, valid_snapshot)

In [15]:
train_features.shape

(4266, 13)

In [16]:
train_features.head()

,Customer ID,first_purchase,last_purchase,frequency,monetary,recency,lifespan,avg_order_value,active_months,avg_purchase_gap,spending_volatility,avg_basket_size,revenue_contribution
0,12346,2009-12-14 08:34:00,2010-06-28 13:53:00,11,372.86,156,196,33.896364,4,19.2,37.302802,6.363636,0.000044
1,12347,2010-10-31 14:20:00,2010-10-31 14:20:00,1,611.53,31,0,611.530000,1,0.0,0.000000,509.000000,0.000072
2,12348,2010-09-27 14:59:00,2010-09-27 14:59:00,1,222.16,65,0,222.160000,1,0.0,0.000000,373.000000,0.000026
3,12349,2010-04-29 13:20:00,2010-10-28 08:23:00,3,2671.14,34,181,890.380000,3,90.0,620.785076,331.000000,0.000315
4,12351,2010-11-29 15:23:00,2010-11-29 15:23:00,1,300.93,2,0,300.930000,1,0.0,0.000000,261.000000,0.000035


## Save Train & Validation Feature Files

In [17]:
train_features.to_csv("../data/processed/train_customer_features.csv", index=False)

valid_features.to_csv("../data/processed/valid_customer_features.csv", index=False)

## Notebook 03 – Feature Engineering Summary  

In this notebook, I transformed the cleaned transactional dataset into a structured customer-level dataset suitable for segmentation analysis.

The primary objective was to design a reusable feature engineering pipeline that captures meaningful customer behavior while maintaining temporal consistency between training and validation periods.

---

### Key Steps Performed

- Built a reusable function: `build_customer_features()`
- Converted transaction-level data into customer-level aggregates
- Applied separate snapshot dates for train and validation datasets
- Generated structured feature tables:
  - `train_features`
  - `valid_features`
- Saved final feature datasets to `/data/processed/`

---

### Features Engineered

In addition to classical RFM metrics, several advanced behavioral and temporal features were constructed:

- **Recency** – Days since last purchase  
- **Frequency** – Number of unique invoices  
- **Monetary** – Total revenue generated  
- **Customer Lifespan** – Duration between first and last purchase  
- **Average Order Value**  
- **Active Months** – Number of months with purchases  
- **Average Purchase Gap** – Mean days between invoices (calculated at invoice level to avoid line-item bias)  
- **Spending Volatility** – Standard deviation of invoice totals  
- **Average Basket Size** – Mean quantity per invoice  
- **Revenue Contribution** – Share of total revenue  

Special attention was given to computing purchase regularity and volatility at the **invoice level**, ensuring that multi-line invoices did not artificially distort temporal metrics.

---

###  Design Principle

This notebook focuses strictly on **feature construction**.

No modeling-related steps such as scaling, outlier handling, PCA, or clustering were applied here.  
Those steps will be performed separately in Notebook 04 to maintain a clean separation between:

- **Customer behavior representation**
- **Model preparation and clustering**

---

This concludes the feature engineering phase. The resulting datasets provide a comprehensive behavioral representation of customers and are ready for preprocessing and clustering in the next stage.